# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hisham-Walid/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This notebook translates the provisional CTR / Engagement Opportunity lane into a testable ML task. The framing keeps the decision and content action ahead of the model.

## 1. My lane as an ML task (type)

My lane is **ranking / priority scoring**, not simple classification. For a content strategist deciding which pages to inspect first, the system will assign each eligible page a continuous opportunity score and return a capacity-limited review queue. A higher score means the page appears to have more recoverable click opportunity relative to comparable pages at a similar search position; it does **not** mean a rewrite is automatically justified. Ranking matches the real decision because the team needs an ordered shortlist, not a yes/no prediction for every page.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_SEED = 42
MIN_VISIBLE_IMPRESSIONS = 100
REVIEW_CAPACITY = 20
LANE = "CTR / Engagement Opportunity Scoring"
TASK_TYPE = "ranking / priority scoring"

root = Path.cwd()
while root != root.parent and not (root / "data/raw/content_refresh_anonymized.csv").exists():
    root = root.parent
data_path = root / "data/raw/content_refresh_anonymized.csv"
assert data_path.exists(), "Run this notebook from inside the starter repository."

df = pd.read_csv(data_path)
lane_slice = df[(df["impressions_90d"] >= MIN_VISIBLE_IMPRESSIONS) & (df["avg_position"] > 0)].copy()
assert lane_slice["content_id"].is_unique

print(f"Lane: {LANE}")
print(f"Task type: {TASK_TYPE}")
print(f"Eligible page rows: {len(lane_slice):,}")


Lane: CTR / Engagement Opportunity Scoring
Task type: ranking / priority scoring
Eligible page rows: 22,006


## 2. Target or proxy

The intended continuous target is **future missed-click opportunity** for the next 30-day outcome window. For each page, I will measure its observed future CTR and compare it with the future CTR expected for peers in the same position and content strata. The target is `future_missed_clicks = future_impressions × max(peer_expected_future_ctr − observed_future_ctr, 0) / 100`. Both impressions and CTR come from a later, observed window; the peer expectation will be estimated without using held-out clients.

The starter CSV is only one trailing-90-day snapshot, so it cannot provide that future label. The code below creates `current_missed_clicks_proxy` from the current snapshot solely to make the target's scale and direction concrete. It is a descriptive proxy, **not** a training label or validation result. The warehouse model will use strictly earlier feature windows and later outcome windows.

In [2]:
peer_keys = ["position_tier", "content_type"]
lane_slice["peer_median_ctr_pct"] = lane_slice.groupby(peer_keys, observed=True)["ctr"].transform("median")
lane_slice["current_ctr_gap_pp"] = (lane_slice["peer_median_ctr_pct"] - lane_slice["ctr"]).clip(lower=0)
lane_slice["current_missed_clicks_proxy"] = (
    lane_slice["impressions_90d"] * lane_slice["current_ctr_gap_pp"] / 100
)

target_sketch = pd.DataFrame({
    "column": ["future_impressions", "future_observed_ctr_pct", "peer_expected_future_ctr_pct", "future_missed_clicks"],
    "source": ["next 30 days", "next 30 days", "out-of-fold peer estimate", "calculated from observed future outcomes"],
    "available_in_starter_snapshot": [False, False, False, False],
})
display(target_sketch)
print(f"Current proxy median: {lane_slice['current_missed_clicks_proxy'].median():.2f} missed clicks")
print(f"Current proxy 90th percentile: {lane_slice['current_missed_clicks_proxy'].quantile(0.90):.2f} missed clicks")


,column,source,available_in_starter_snapshot
0,future_impressions,next 30 days,False
1,future_observed_ctr_pct,next 30 days,False
2,peer_expected_future_ctr_pct,out-of-fold peer estimate,False
3,future_missed_clicks,calculated from observed future outcomes,False


Current proxy median: 0.00 missed clicks
Current proxy 90th percentile: 3.99 missed clicks


## 3. Success metric

The primary metric is **NDCG@20**, computed within each held-out client and summarized by the median across clients. It rewards putting pages with more observed future missed-click opportunity near the top, respects the editor's review capacity of 20 pages, and uses the continuous target without inventing a yes/no label. Before training, I define “good” as median client NDCG@20 of at least **0.60** and at least **10% relative improvement** over a transparent lagged, position-adjusted CTR rule on the same client-held-out, forward-time evaluation. I will also report the distribution across clients so a strong average cannot hide weak segments.

In [3]:
def ndcg_at_k(relevance, scores, k=20):
    relevance = np.asarray(relevance, dtype=float)
    scores = np.asarray(scores, dtype=float)
    k = min(k, len(relevance))
    discounts = np.log2(np.arange(2, k + 2))
    ranked = relevance[np.argsort(-scores, kind="stable")[:k]]
    ideal = np.sort(relevance)[::-1][:k]
    dcg = np.sum(ranked / discounts)
    idcg = np.sum(ideal / discounts)
    return dcg / idcg if idcg > 0 else np.nan

metric_demo = lane_slice["current_missed_clicks_proxy"].to_numpy()
rng = np.random.default_rng(RANDOM_SEED)
oracle_ndcg = ndcg_at_k(metric_demo, metric_demo, REVIEW_CAPACITY)
shuffled_ndcg = ndcg_at_k(metric_demo, rng.permutation(metric_demo), REVIEW_CAPACITY)
assert np.isclose(oracle_ndcg, 1.0)
print(f"Metric implementation check — oracle NDCG@20: {oracle_ndcg:.3f}")
print(f"Illustrative seeded shuffle NDCG@20: {shuffled_ndcg:.3f}")
print("These values test metric mechanics only; they are not model results.")


Metric implementation check — oracle NDCG@20: 1.000
Illustrative seeded shuffle NDCG@20: 0.001
These values test metric mechanics only; they are not model results.


## 4. The unit of analysis, as a real dataframe

**One row is one pseudonymized content page** with at least 100 impressions and valid position data in the starter window. The preview deliberately omits identifiers. At warehouse scale, every model row will be a page at a feature-window cutoff, with historical features before the cutoff and the target measured in the following 30 days. Client IDs remain available only for grouped splits and joins, never as features. Missing keyword and word-count values receive explicit availability flags rather than a blind zero fill.

In [4]:
lane_slice["has_keyword_context"] = lane_slice["search_volume"].notna()
lane_slice["has_word_count"] = lane_slice["word_count"].notna()
lane_slice["future_missed_clicks"] = np.nan  # Requires the later warehouse outcome window.

unit_columns = [
    "impressions_90d", "avg_position", "ctr", "content_type", "main_intent",
    "content_age_days", "days_since_last_update", "has_keyword_context",
    "has_word_count", "current_missed_clicks_proxy", "future_missed_clicks",
]
unit_frame = lane_slice[unit_columns].sort_values("current_missed_clicks_proxy", ascending=False).head(8)
display(unit_frame.style.format({"ctr": "{:.3f}", "current_missed_clicks_proxy": "{:.1f}"}))
print(f"Displayed shape: {unit_frame.shape}; full eligible slice: {lane_slice.shape[0]:,} unique pages.")


,impressions_90d,avg_position,ctr,content_type,main_intent,content_age_days,days_since_last_update,has_keyword_context,has_word_count,current_missed_clicks_proxy,future_missed_clicks
3394,295097,7.300000,0.050,keyword article,informational,144,104,True,False,531.2,nan
7445,208678,9.700000,0.000,keyword article,informational,362,104,True,False,480.0,nan
6653,517715,4.200000,0.140,keyword article,informational,537,104,True,False,465.9,nan
6903,223271,7.800000,0.030,keyword article,informational,95,20,True,True,446.5,nan
7678,272144,2.300000,0.030,keyword article,informational,280,20,True,True,435.4,nan
27178,140079,7.600000,0.010,keyword article,informational,97,20,True,True,308.2,nan
22028,213963,4.700000,0.100,keyword article,informational,97,20,True,True,278.2,nan
3070,159590,7.800000,0.060,keyword article,commercial,257,104,True,False,271.3,nan


Displayed shape: (8, 11); full eligible slice: 22,006 unique pages.


## 5. Why ML beats a fixed rule here

A fixed rule such as `CTR < 0.5%` ignores how expected CTR changes with position, volume, content type, intent, age, and freshness. It also treats a one-click page like a high-impression page. The audit below shows that the same global cutoff produces very different flag rates across position tiers, so it is not a fair priority score. A learned ranker may capture nonlinear interactions and uncertainty more consistently, but it earns deployment only if forward-time, client-held-out evaluation beats the transparent position-adjusted baseline. If it does not, the rule remains the better system.

For a content strategist deciding which 20 pages to inspect, the output supports a human review of intent match, title/meta wording, snippet structure, and on-page engagement. The project claims only observed associations and directional decision support—not that an edit will cause traffic growth or that the model explains a search engine's algorithm.

In [5]:
lane_slice["global_low_ctr_rule"] = lane_slice["ctr"] < 0.5  # 0.5 means 0.5%.
rule_audit = (
    lane_slice.groupby("position_tier", observed=True)
    .agg(
        pages=("content_id", "size"),
        mean_ctr_pct=("ctr", "mean"),
        pct_flagged_by_global_rule=("global_low_ctr_rule", "mean"),
    )
    .sort_values("mean_ctr_pct", ascending=False)
)
rule_audit["pct_flagged_by_global_rule"] *= 100
display(rule_audit.style.format({"mean_ctr_pct": "{:.3f}", "pct_flagged_by_global_rule": "{:.1f}%"}))

planned_features = {
    "prior_impressions", "prior_avg_position", "prior_ctr", "content_type",
    "main_intent", "content_age_days", "days_since_last_update",
    "has_keyword_context", "has_word_count",
}
forbidden = {"content_id", "client_id", "trend_pct", "trend_direction", "future_missed_clicks"}
assert planned_features.isdisjoint(forbidden)
print("Leakage check passed: identifiers, current trend-label fields, and the future outcome are excluded.")


,pages,mean_ctr_pct,pct_flagged_by_global_rule
position_tier,,,
page_1,8633,0.355,77.8%
top_3,533,0.334,76.5%
striking,5903,0.256,84.4%
page_3_5,6058,0.142,93.5%
deep,879,0.055,96.4%


Leakage check passed: identifiers, current trend-label fields, and the future outcome are excluded.


## Self-check

Before submitting, I verified:

- [x] Every section above is filled with written reasoning and supporting code
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries are displayed
- [x] Claims use careful words: observed, measured, directional, decision-support
- [x] The completed notebook is under `work/notebooks/` and ready to commit